# Etape 0 _ GDELT × Bénin : Pipeline de nettoyage reproductible
**Hackathon iSHEERO × DataCamp 2026 : Bénin Insights Challenge**

Ce notebook documente et reproduit toutes les étapes de nettoyage des données GDELT  
pour isoler les événements concernant **le pays Bénin** (et non Benin City, Nigeria).

GDELT intègre dans ses données relatives au Bénin des événements localisés à Benin City, ville du Nigéria, en raison de la similarité des noms. Ces entrées ont été identifiées et exclues grâce à un filtre combinant le Filtrage des faux positifs nigérians <a id='4-filtre1'></a> et l'Exclusion des géocodages nigérians explicites <a id='5-filtre2'></a>. Ce biais, s'il n'est pas corrigé, gonfle artificiellement le volume d'événements attribués au Bénin et introduit des données nigérianes dans l'analyse.

**Auteurs :** *Fidele*  
**Date :** 2025  
**Sources :** GDELT Project `GDELT_events_benin_2025.csv`


## Sommaire
1. [Imports et configuration](#1-imports)
2. [Chargement des données](#2-chargement)
3. [Diagnostic du problème Benin City / Pays Bénin](#3-diagnostic)
4. [D 1 — Filtrage des faux positifs nigérians](#4-filtre1)
5. [D 2 — Exclusion des géocodages nigérians explicites](#5-filtre2)
6. [Bilan du nettoyage](#6-bilan)
7. [Traitement des valeurs manquantes](#7-missing)
8. [Export du dataset propre](#8-export)


## 1. Imports et configuration <a id='1-imports'></a>

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Chemins (adapter si nécessaire) 
INPUT_FILE  = "GDELT_Challenge_Isheero_2026_Team09\data\GDELT_events_benin_2025.csv"   # fichier brut GDELT

OUTPUT_FILE = "GDELT_Challenge_Isheero_2026_Team09\data\GDELT_benin_clean.csv"          # fichier nettoyé produit

print(" Imports OK")
print(f"   Fichier source  : {INPUT_FILE}")
print(f"   Fichier cible   : {OUTPUT_FILE}")


 Imports OK
   Fichier source  : GDELT_events_benin_2025.csv
   Fichier cible   : GDELT_benin_clean.csv


## 2. Chargement des données <a id='2-chargement'></a>

On charge le fichier brut extrait de GDELT pour le Bénin.  
GDELT encode certaines colonnes avec des conventions propres (codes FIPS, ISO-3, etc.).


In [2]:
df = pd.read_csv(INPUT_FILE, low_memory=False)

# Extraction du domaine source depuis l'URL
df['domain'] = df['SOURCEURL'].str.extract(r'https?://(?:www\.)?([^/]+)')

print(f"Shape brut : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print()
print("Aperçu des colonnes disponibles :")
print(df.dtypes.to_string())


Shape brut : 34,106 lignes × 27 colonnes

Aperçu des colonnes disponibles :
Unnamed: 0                 int64
GLOBALEVENTID              int64
SQLDATE                    int64
MonthYear                  int64
Actor1Name                object
Actor1CountryCode         object
Actor1Type1Code           object
Actor2Name                object
Actor2CountryCode         object
Actor2Type1Code           object
IsRootEvent                int64
EventCode                  int64
EventBaseCode              int64
EventRootCode              int64
QuadClass                  int64
GoldsteinScale           float64
NumMentions                int64
NumSources                 int64
NumArticles                int64
AvgTone                  float64
ActionGeo_FullName        object
ActionGeo_CountryCode     object
ActionGeo_ADM1Code        object
ActionGeo_Lat            float64
ActionGeo_Long           float64
SOURCEURL                 object
domain                    object


In [3]:
# Aperçu rapide
df[['Actor1Name', 'Actor1CountryCode', 'Actor2Name', 'Actor2CountryCode',
    'ActionGeo_FullName', 'ActionGeo_CountryCode', 'GoldsteinScale',
    'QuadClass', 'SOURCEURL']].head(5)


,Actor1Name,Actor1CountryCode,Actor2Name,Actor2CountryCode,ActionGeo_FullName,ActionGeo_CountryCode,GoldsteinScale,QuadClass,SOURCEURL
0,AL QAEDA,NaN,BENIN,BEN,Benin,BN,-10.0,4,https://www.abc.com.py/internacionales/2026/01...
1,NaN,NaN,NIGERIAN,NGA,"Borgou, Borgou, Benin",BN,-10.0,4,https://allafrica.com/stories/202603090010.html
2,PRISONER,NaN,NaN,NaN,Benin,BN,3.2,1,https://lanouvelletribune.info/2026/03/benin-d...
3,BENIN,BEN,NaN,NaN,Benin,BN,2.8,1,https://www.rewmi.com/koulibaly-la-rdc-va-fair...
4,COMMANDANT,NaN,NaN,NaN,Benin,BN,0.0,1,https://dailypost.ng/2025/12/31/nscdc-arrests-...


## 3. Diagnostic : Problème Benin City / Pays Bénin <a id='3-diagnostic'></a>

### Contexte du problème
GDELT collecte des articles mentionnant le mot **"Benin"** dans le texte.  
Or ce mot désigne deux réalités géographiques distinctes :

| Entité | Pays | Coordonnées approximatives |
|--------|------|---------------------------|
| **Pays Bénin** | Bénin | 9.5°N, 2.25°E |
| **Benin City** | Nigeria (Edo State) | 6.34°N, 5.63°E |

GDELT géocode **correctement** au centroïde du pays (9.5°N, 2.25°E),  
mais **ingère à tort** des articles nigérians traitant de Benin City.  
Ces articles font gonfler artificiellement le volume et biaisent les sources.

>  **Note technique :** GDELT utilise deux systèmes de codes pays différents :
> - `Actor1/2CountryCode` → **ISO Alpha-3** (ex: `BEN`, `NGA`)  
> - `ActionGeo_CountryCode` → **FIPS** (ex: `BN`, `NI`)


In [18]:
# ── Distribution des sources 
print("=== Top 20 domaines sources (brut) ===")
print(df['domain'].value_counts().head(20).to_string())


=== Top 20 domaines sources (brut) ===
domain
punchng.com                    1976
dailypost.ng                   1879
nigerianobservernews.com       1204
allafrica.com                  1142
leadership.ng                  1041
guardian.ng                     895
lanouvelletribune.info          877
thisdaylive.com                 748
saharareporters.com             697
thesun.ng                       674
premiumtimesng.com              654
blueprint.ng                    623
quicknews-africa.net            570
thenationonlineng.net           517
dailytrust.com                  495
thecable.ng                     452
theeagleonline.com.ng           448
tribuneonlineng.com             422
yahoo.com                       338
nationalaccordnewspaper.com     333


In [ ]:
# ── Vérification des coordonnées GPS pour ActionGeo_FullName == 'Benin' 
benin_geo = df[df['ActionGeo_FullName'] == 'Benin']
print(f"Lignes avec ActionGeo_FullName == 'Benin' : {len(benin_geo):,}")
print()
print("Coordonnées GPS (toutes identiques ?):")
print(benin_geo[['ActionGeo_Lat', 'ActionGeo_Long']].describe())
print()
print("→ GDELT géocode bien au centroïde du PAYS Bénin (9.5°N, 2.25°E)")
print("  Le problème vient du contenu des articles, pas du géocodage GPS.")


Lignes avec ActionGeo_FullName == 'Benin' : 21,758

Coordonnées GPS (toutes identiques ?):
       ActionGeo_Lat  ActionGeo_Long
count        21758.0        21758.00
mean             9.5            2.25
std              0.0            0.00
min              9.5            2.25
25%              9.5            2.25
50%              9.5            2.25
75%              9.5            2.25
max              9.5            2.25

→ GDELT géocode bien au centroïde du PAYS Bénin (9.5°N, 2.25°E)
  Le problème vient du contenu des articles, pas du géocodage GPS.


In [17]:
# ── Identifier les sources nigérianes
NIGERIAN_DOMAINS_PATTERN = (
    r'\.ng$|punchng|nigerianobserver|thisdaylive|saharareporters|'
    r'premiumtimesng|thenationonlineng|nationalguideng|dailytrust|'
    r'naijanews|nigerianeye|withinnigeria|nigeriasun|vanguardngr|'
    r'channelstv|opinionnigeria|informationng|thenewsnigeria|'
    r'tribuneonlineng|theeagleonline|blueprint\.ng|thecable\.ng|'
    r'legit\.ng|naija247|pulse\.ng|thenigerianvoice|bellanaija|'
    r'nationalaccord|prompnewsonline|thesun\.ng|guardian\.ng|'
    r'leadership\.ng|quicknews|myjoyonline'
)

is_nigerian_source = df['domain'].str.contains(
    NIGERIAN_DOMAINS_PATTERN, case=False, na=False
)

print(f"Articles de sources nigérianes dans le brut : {is_nigerian_source.sum():,}")
print(f"Soit {100*is_nigerian_source.mean():.1f}% du total")


Articles de sources nigérianes dans le brut : 16,373
Soit 48.0% du total


## 4. D 1 : Filtrage des faux positifs nigérians <a id='4-filtre1'></a>

### Logique du filtre
Un article est considéré comme **faux positif nigérian** s'il réunit **les 3 conditions** :

1. **Source nigériane** identifiée (domaine `.ng` ou média nigérian connu)
2. **URL contenant des mots-clés géographiques nigérians** (edo, lagos, abuja, kano...)
3. **Géocodage générique** `ActionGeo_FullName == 'Benin'` (pas de ville béninoise spécifique)

> La condition 3 protège contre le sur-filtrage : un article nigérian qui  
> mentionne explicitement une ville béninoise (ex: Cotonou, Parakou) est conservé.


In [ ]:
# ── Mots-clés géographiques nigérians dans l'URL 
NIGERIA_URL_KEYWORDS = (
    r'edo|benin-city|benin_city|edo-state|nigeria|nigerian|'
    r'abuja|lagos|ogun|oyo|kano|kaduna|rivers|delta|anambra|'
    r'imo|enugu|cross-river|akwa|bayelsa|ondo|ekiti|osun|'
    r'kwara|benue|plateau|nasarawa|kogi|niger-state|zamfara|'
    r'sokoto|kebbi|jigawa|bauchi|gombe|yobe|borno|adamawa|taraba'
)

has_nigeria_url_kw = df['SOURCEURL'].str.contains(
    NIGERIA_URL_KEYWORDS, case=False, na=False
)

# ── Villes béninoises spécifiques dans ActionGeo
BENIN_SPECIFIC_PLACES = (
    r'Ouidah|Porto.Novo|Parakou|Cotonou|Abomey|Lokossa|Natitingou|'
    r'Djougou|Kandi|Borgou|Alibori|Atacora|Atakora|Donga|Collines|'
    r'Mono|Couffo|Zou|Atlantique|Tchaourou|Malanville|Porga|Karimama|'
    r'Tanguieta|Savalou|Allada|Godomey|Ganvie|Pobe|Kalale|Ketu|'
    r'Ndali|Kpodji|Tchoukou|Koudou'
)

has_benin_specific_geo = df['ActionGeo_FullName'].str.contains(
    BENIN_SPECIFIC_PLACES, case=False, na=False
)

is_generic_benin = df['ActionGeo_FullName'] == 'Benin'

# ── Construction du masque faux positifs 
is_false_positive = (
    is_nigerian_source       # source nigériane
    & has_nigeria_url_kw     # URL avec mots-clés Nigeria
    & is_generic_benin       # géo générique "Benin"
    & ~has_benin_specific_geo  # pas de ville béninoise spécifique
)

print(f"Faux positifs nigérians identifiés (Filtre 1) : {is_false_positive.sum():,}")
print(f"Soit {100*is_false_positive.mean():.1f}% du total brut")
print()
print("Exemples d'URLs filtrées :")
for url in df[is_false_positive]['SOURCEURL'].head(5).values:
    print(f"  {url}")


Faux positifs nigérians identifiés (Filtre 1) : 7,471
Soit 21.9% du total brut

Exemples d'URLs filtrées :
  https://dailypost.ng/2025/12/31/nscdc-arrests-suspected-oil-installation-vandal-in-edo/
  https://dailypost.ng/2025/12/31/nscdc-arrests-suspected-oil-installation-vandal-in-edo/
  https://dailypost.ng/2025/12/31/nscdc-arrests-suspected-oil-installation-vandal-in-edo/
  https://dailypost.ng/2025/12/31/nscdc-arrests-suspected-oil-installation-vandal-in-edo/
  https://punchng.com/obasekis-ally-house-burnt-in-edo/


## 5. D 2 : Exclusion des géocodages nigérians explicites <a id='5-filtre2'></a>

Certains articles sont géocodés dans des **lieux nigérians explicites**  
(Lagos, Abuja, Edo State, Tinubu...) : ils doivent être exclus indépendamment de la source.

> Exception : si l'article mentionne aussi une ville béninoise spécifique  
> (ex: article transfrontalier), il est conservé.


In [ ]:
# ── Lieux nigérians dans ActionGeo_FullName 
NIGERIAN_PLACES_PATTERN = (
    r'Nigeria|Edo State|Lagos|Abuja|Kano|Delta|Rivers|Ogun|Oyo|'
    r'Ondo|Enugu|Kaduna|Anambra|Imo|Tinubu|Benin City|Edo$'
)

has_nigerian_geo = df['ActionGeo_FullName'].str.contains(
    NIGERIAN_PLACES_PATTERN, case=False, na=False
)

# Exclure seulement si pas de mention béninoise spécifique
is_clearly_nigeria = has_nigerian_geo & ~has_benin_specific_geo

print(f"Géocodages nigérians explicites (Filtre 2) : {is_clearly_nigeria.sum():,}")
print(f"Soit {100*is_clearly_nigeria.mean():.1f}% du total brut")
print()
print("Top ActionGeo_FullName exclus :")
print(df[is_clearly_nigeria]['ActionGeo_FullName'].value_counts().head(10).to_string())


Géocodages nigérians explicites (Filtre 2) : 4,597
Soit 13.5% du total brut

Top ActionGeo_FullName exclus :
ActionGeo_FullName
Edo State, Edo, Nigeria                            873
Nigeria                                            798
Abuja, Abuja Federal Capital Territory, Nigeria    621
Lagos, Lagos, Nigeria                              422
Tinubu, Lagos, Nigeria                             122
London, London, City of, United Kingdom             90
Cross River State, Cross River, Nigeria             67
Ibadan, Oyo, Nigeria                                63
Lake Chad, Nigeria (general), Nigeria               62
Kano, Kano, Nigeria                                 58


## 6. Bilan du nettoyage <a id='6-bilan'></a>

In [ ]:
# ── Application des deux filtres 
mask_exclude = is_false_positive | is_clearly_nigeria
df_clean = df[~mask_exclude].copy()

print("=" * 55)
print("  BILAN DU NETTOYAGE GDELT × BÉNIN")
print("=" * 55)
print(f"  Dataset brut             : {len(df):>7,} événements")
print(f"  Filtre 1 (faux positifs) : -{is_false_positive.sum():>6,}")
print(f"  Filtre 2 (géo Nigeria)   : -{(is_clearly_nigeria & ~is_false_positive).sum():>6,}")
print(f"  Dataset nettoyé          : {len(df_clean):>7,} événements")
print(f"  Réduction totale         : {100*(1 - len(df_clean)/len(df)):.1f}%")
print("=" * 55)
print()
print("Bruit résiduel estimé (contrôle) :")
residual = df_clean['SOURCEURL'].str.contains(
    r'/edo/|/edo-state/|benin-city|benin_city', case=False, na=False
)
print(f"  URLs résiduelles avec 'edo/benin-city' : {residual.sum()} ({100*residual.mean():.2f}%)")


  BILAN DU NETTOYAGE GDELT × BÉNIN
  Dataset brut             :  34,106 événements
  Filtre 1 (faux positifs) : - 7,471
  Filtre 2 (géo Nigeria)   : - 4,597
  Dataset nettoyé          :  22,038 événements
  Réduction totale         : 35.4%

Bruit résiduel estimé (contrôle) :
  URLs résiduelles avec 'edo/benin-city' : 37 (0.17%)


In [ ]:
# ── Top ActionGeo après nettoyage 
print("Top lieux géocodés après nettoyage :")
print(df_clean['ActionGeo_FullName'].value_counts().head(15).to_string())
print()
print("Top domaines sources après nettoyage :")
print(df_clean['domain'].value_counts().head(15).to_string())


Top lieux géocodés après nettoyage :
ActionGeo_FullName
Benin                              14287
France                               425
Ghana                                386
Togo                                 305
Niger                                260
Ouidah, Atlantique, Benin            218
Porto-Novo, Qué, Benin               196
Paris, France (general), France      187
Burkina Faso                         164
Abomey, Zou, Benin                   155
United Kingdom                       128
Niamey, Niamey, Niger                123
Parakou, Borgou, Benin               110
Alibori, Alibori, Benin              109
United States                        108

Top domaines sources après nettoyage :
domain
allafrica.com             986
lanouvelletribune.info    873
punchng.com               713
dailypost.ng              502
leadership.ng             455
guardian.ng               326
yahoo.com                 316
thisdaylive.com           310
thenationonlineng.net     286
saharareport

## 7. Traitement des valeurs manquantes <a id='7-missing'></a>

### Philosophie GDELT
Dans GDELT, les cellules vides sont souvent **intentionnelles** et **sémantiquement significatives**.  
Il ne faut **jamais imputer une valeur numérique** (moyenne, médiane) sur des champs catégoriels.

| Champ | % vide | Nature du vide | Recommandation |
|-------|--------|---------------|----------------|
| `Actor1/2Type1Code` | 62–73% | Convention : entité géopolitique sans type fonctionnel | Créer catégorie `STATE` ou `UNKNOWN` |
| `Actor2Name` | ~25% | Événement unilatéral (déclaration, action sans cible) | Conserver : informatif |
| `Actor2CountryCode` | ~46% | Acteur fonctionnel (GOVERNMENT, POLICE…) | Imputation contextuelle possible |
| `ActionGeo_*` | ~0.2% | Anomalie de géocodage | Supprimer si analyse géospatiale |


In [ ]:
# ── État des valeurs manquantes dans le dataset nettoyé
missing = df_clean.isnull().sum()
missing_pct = (missing / len(df_clean) * 100).round(1)
summary = pd.DataFrame({'Manquants': missing, '%': missing_pct})
summary = summary[summary['Manquants'] > 0].sort_values('%', ascending=False)
print("Valeurs manquantes dans le dataset nettoyé :")
print(summary.to_string())


Valeurs manquantes dans le dataset nettoyé :
                       Manquants     %
Actor2Type1Code            16054  72.8
Actor1Type1Code            13703  62.2
Actor2CountryCode          10170  46.1
Actor1CountryCode           8275  37.5
Actor2Name                  5618  25.5
Actor1Name                  1844   8.4
ActionGeo_FullName            37   0.2
ActionGeo_CountryCode         37   0.2
ActionGeo_ADM1Code            37   0.2
ActionGeo_Lat                 37   0.2
ActionGeo_Long                37   0.2


In [ ]:
# ── Imputation 1 : Actor1/2Type1Code 
# Si le type est vide et le nom est un pays connu → 'STATE'
# Sinon → 'UNKNOWN' (entité locale non classifiée)

KNOWN_COUNTRIES = {
    'BENIN', 'NIGERIA', 'GHANA', 'TOGO', 'NIGER', 'FRANCE', 'USA',
    'SENEGAL', 'AFRICA', 'MALI', 'BURKINA', 'CAMEROON', 'COTE',
    'IVORY', 'CHAD', 'CHINESE', 'RUSSIA', 'GERMANY', 'EUROPEAN'
}

for col_type, col_name in [('Actor1Type1Code', 'Actor1Name'),
                             ('Actor2Type1Code', 'Actor2Name')]:
    df_clean[col_type + '_filled'] = df_clean[col_type].copy()
    mask_state = (
        df_clean[col_type].isna()
        & df_clean[col_name].str.upper().isin(KNOWN_COUNTRIES)
    )
    df_clean.loc[mask_state, col_type + '_filled'] = 'STATE'
    df_clean[col_type + '_filled'] = df_clean[col_type + '_filled'].fillna('UNKNOWN')

print("Actor1Type1Code_filled :")
print(df_clean['Actor1Type1Code_filled'].value_counts().head(8).to_string())
print()
print("Actor2Type1Code_filled :")
print(df_clean['Actor2Type1Code_filled'].value_counts().head(8).to_string())


Actor1Type1Code_filled :
Actor1Type1Code_filled
STATE      8040
UNKNOWN    5663
GOV        3135
MIL         726
IGO         558
CVL         465
COP         451
EDU         423

Actor2Type1Code_filled :
Actor2Type1Code_filled
UNKNOWN    8936
STATE      7118
GOV        2163
MIL         671
CVL         415
MED         391
EDU         350
BUS         285


In [14]:
# ── Imputation 2 : Actor2CountryCode pour acteurs institutionnels ────────
# Si Actor2 est une institution (GOVERNMENT, POLICE…) et que l'action
# se déroule au Bénin (FIPS: BN) → imputer 'BEN' (ISO-3)

INSTITUTIONAL_ACTORS = {
    'GOVERNMENT', 'PRESIDENT', 'MILITARY', 'GOVERNOR',
    'MINIST', 'POLICE', 'COURT', 'PARLIAMENT'
}

mask_gov_benin = (
    df_clean['Actor2CountryCode'].isna()
    & df_clean['Actor2Name'].isin(INSTITUTIONAL_ACTORS)
    & (df_clean['ActionGeo_CountryCode'] == 'BN')
)

df_clean.loc[mask_gov_benin, 'Actor2CountryCode'] = 'BEN'

print(f"Actor2CountryCode imputé à 'BEN' (institutions béninoises) : {mask_gov_benin.sum()} lignes")


Actor2CountryCode imputé à 'BEN' (institutions béninoises) : 1046 lignes


In [ ]:
# ── Sous-datasets recommandés selon l'usage analytique

# 1. Analyse géospatiale : exclure les 37 lignes sans coordonnées
df_geo = df_clean[df_clean['ActionGeo_Lat'].notna()].copy()
print(f"Dataset géospatial  : {len(df_geo):,} événements (ActionGeo non nul)")

# 2. Analyse réseau acteurs : exclure les événements unilatéraux
df_network = df_clean[df_clean['Actor2Name'].notna()].copy()
print(f"Dataset réseau      : {len(df_network):,} événements (Actor2 présent)")

# 3. Dataset complet nettoyé (usage général)
print(f"Dataset complet     : {len(df_clean):,} événements")


Dataset géospatial  : 22,001 événements (ActionGeo non nul)
Dataset réseau      : 16,420 événements (Actor2 présent)
Dataset complet     : 22,038 événements


## 8. Export du dataset propre <a id='8-export'></a>

In [19]:
# ── Export principal 
df_clean.to_csv(OUTPUT_FILE, index=False)
print(f" Dataset nettoyé exporté : {OUTPUT_FILE}")
print(f"   {len(df_clean):,} lignes × {df_clean.shape[1]} colonnes")

# ── Exports optionnels 
# df_geo.to_csv("GDELT_benin_geo.csv", index=False)       # analyse géospatiale
# df_network.to_csv("GDELT_benin_network.csv", index=False) # analyse réseau

print()
print("Colonnes du dataset nettoyé :")
print(list(df_clean.columns))


 Dataset nettoyé exporté : GDELT_benin_clean.csv
   22,038 lignes × 29 colonnes

Colonnes du dataset nettoyé :
['Unnamed: 0', 'GLOBALEVENTID', 'SQLDATE', 'MonthYear', 'Actor1Name', 'Actor1CountryCode', 'Actor1Type1Code', 'Actor2Name', 'Actor2CountryCode', 'Actor2Type1Code', 'IsRootEvent', 'EventCode', 'EventBaseCode', 'EventRootCode', 'QuadClass', 'GoldsteinScale', 'NumMentions', 'NumSources', 'NumArticles', 'AvgTone', 'ActionGeo_FullName', 'ActionGeo_CountryCode', 'ActionGeo_ADM1Code', 'ActionGeo_Lat', 'ActionGeo_Long', 'SOURCEURL', 'domain', 'Actor1Type1Code_filled', 'Actor2Type1Code_filled']


In [20]:
# ## Récapitulatif des décisions de nettoyage

#| Étape | Décision | Justification |
#|-------|----------|--------------|
#| Filtre 1 | Supprimer sources `.ng` + URL nigeria + géo "Benin" générique | Faux positifs Benin City/Nigeria |
#| Filtre 2 | Supprimer géocodages explicitement nigérians | Articles hors-périmètre géographique |
#| Vides Actor Type | Imputer `STATE` si pays connu, `UNKNOWN` sinon | Convention GDELT documentée |
#| Vides Actor2 | Conserver — événements unilatéraux valides | Perte d'info si suppression |
#| Vides ActorCountry | Imputer `BEN` pour institutions en ActionGeo BN | Inférence contextuelle fiable |
#| Vides ActionGeo | Supprimer si analyse géospatiale uniquement | 37 lignes, anomalies résiduelles |

#**Bruit résiduel estimé :** < 0.3% du dataset nettoyé.
